In [1]:
import os
import sys
import gc
import glob
import joblib
import numpy as np
import pandas as pd
import warnings
import torch
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

In [2]:
SRC_DIR = '/kaggle/input/datasets/thnhvinhs/optiver-src-code/finance_time_series - Copy'
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

In [3]:
from src.data.data_loader import DataBlock
from src.features.feature_pipeline import build_features
from src.models.cnn_model import CNN1DModel
from src.models.mlp_model import MLPModel

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Bắt đầu quá trình Inference trên: {device}")

# 2. 🗂️ ĐỌC DỮ LIỆU KIỂM THỬ (TEST SET) BÍ MẬT CỦA KAGGLE
DATA_DIR = '/kaggle/input/competitions/optiver-realized-volatility-prediction'
test_csv_path = os.path.join(DATA_DIR, 'test.csv')

df_test = pd.read_csv(test_csv_path)

# Tạo row_id chuẩn xác theo format Kaggle yêu cầu
df_test['row_id'] = df_test['stock_id'].astype(str) + '-' + df_test['time_id'].astype(str)
print(f"Kích thước tập Test ban đầu: {df_test.shape}")

🚀 Bắt đầu quá trình Inference trên: cpu
Kích thước tập Test ban đầu: (3, 3)


In [5]:
print("⚖️ Đang khôi phục hệ quy chiếu StandardScaler từ tập Train...")
df_train = pd.read_feather('/kaggle/input/notebooks/thnhvinhs/notebook22d94af6ba/Realized-Volatility-Prediction/data/processed/features_FINAL.feather')

features = [col for col in df_train.columns if col not in ['row_id', 'time_id', 'stock_id', 'target', 'true_time_id', 'tsne_val']]

# Fill NaN bằng chuẩn của Train
df_train[features] = df_train[features].replace([np.inf, -np.inf], np.nan).fillna(df_train[features].mean())

# FIT SCALER VÀ LƯU LẠI
train_scaler = StandardScaler()
train_scaler.fit(df_train[features].values)

# NUKE TẬP TRAIN ĐỂ CỨU 10GB RAM
del df_train
gc.collect()
print("✅ Đã chốt Scaler và xả sạch RAM Train. Không lo OOM!")

⚖️ Đang khôi phục hệ quy chiếu StandardScaler từ tập Train...
✅ Đã chốt Scaler và xả sạch RAM Train. Không lo OOM!


In [6]:
print("\nĐang khởi động cỗ máy trích xuất K-NN Features cho Test Set...")
df_features = build_features(
    df_train=df_test, 
    data_dir=DATA_DIR, 
    block=DataBlock.TEST, 
    config_path='/kaggle/input/datasets/thnhvinhs/optiver-src-code/finance_time_series - Copy/configs/feature_config.yaml'
)

# Phân định cột đặc trưng
not_features = ['row_id', 'time_id', 'stock_id', 'target', 'true_time_id', 'tsne_val']
features = [col for col in df_features.columns if col not in not_features]
print(f"Đã trích xuất thành công {len(features)} đặc trưng.")


Đang khởi động cỗ máy trích xuất K-NN Features cho Test Set...
2026-06-09 15:07:24 | INFO     | feature_pipeline | Đã phát hiện 1 unique stock_id trong df_train. Tiến hành filter dữ liệu đọc vào RAM.
2026-06-09 15:07:24 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Bắt đầu...
2026-06-09 15:07:24 | INFO     | data_loader | Đang đọc song song 1 file book test (n_jobs=4)...
2026-06-09 15:07:25 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (3, 11)
2026-06-09 15:07:25 | INFO     | data_loader | Đang đọc song song 1 file trade test (n_jobs=4)...
2026-06-09 15:07:25 | INFO     | data_loader | Hoàn tất đọc dữ liệu. Kích thước tổng: (3, 6)
2026-06-09 15:07:25 | INFO     | feature_pipeline | [a) Đọc dữ liệu Book và Trade song song] Hoàn thành trong 1.2190 giây.
2026-06-09 15:07:25 | INFO     | feature_pipeline | [b) Tính toán Base Features và Tick Size] Bắt đầu...
2026-06-09 15:07:26 | INFO     | feature_pipeline | [b) Tính toán Base Features và Ti

In [7]:
from torch.utils.data import DataLoader, TensorDataset

# 4. 🧹 TIỀN XỬ LÝ VÀ QUẢN LÝ BỘ NHỚ CHO DEEP LEARNING
df_features[features] = df_features[features].replace([np.inf, -np.inf], np.nan)
df_features[features] = df_features[features].fillna(0)

# Chuyển đổi và XÓA NGAY mảng trung gian để cứu RAM
X_test_raw = df_features[features].values
X_test_scaled = train_scaler.transform(X_test_raw) 
del X_test_raw
gc.collect()

X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
del X_test_scaled
gc.collect()

# DataLoader chống OOM cho Test Set ẩn
test_loader = DataLoader(TensorDataset(X_test_tensor), batch_size=1024, shuffle=False)
input_dim = len(features)

In [8]:
predictions = np.zeros(len(df_features))

# --- A. LIGHTGBM ENSEMBLE ---
lgbm_paths = glob.glob('/kaggle/input/notebooks/thnhvinhs/notebook81d6943bd1/Realized-Volatility-Prediction/models/lgbm/*.pkl')
lgbm_preds = np.zeros(len(df_features))
if lgbm_paths:
    print(f"\\n🌳 Đang gọi {len(lgbm_paths)} mô hình LightGBM...")
    for p in lgbm_paths:
        model = joblib.load(p)
        lgbm_preds += model.predict(df_features[features])
    lgbm_preds /= len(lgbm_paths)

# --- B. MLP ENSEMBLE ---
mlp_paths = glob.glob('/kaggle/input/notebooks/thnhvinhs/1d-cnn-and-mlp-with-ensemble/Realized-Volatility-Prediction/models/mlp/*.pth')
mlp_preds = np.zeros(len(df_features))
if mlp_paths:
    print(f"⚡ Đang gọi {len(mlp_paths)} mô hình MLP...")
    for p in mlp_paths:
        model = MLPModel(num_features=input_dim).to(device)
        model.load_state_dict(torch.load(p, map_location=device))
        model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader: # <--- CHẠY TỪNG LÔ NHỎ ĐỂ CHỐNG TRÀN RAM
                preds = model(batch[0]).cpu().numpy()
                fold_preds.append(preds.flatten())
                
        mlp_preds += np.concatenate(fold_preds)
    mlp_preds /= len(mlp_paths)

# --- C. 1D-CNN ENSEMBLE ---
cnn_paths = glob.glob('/kaggle/input/notebooks/thnhvinhs/1d-cnn-and-mlp-with-ensemble/Realized-Volatility-Prediction/models/cnn/*.pth')
cnn_preds = np.zeros(len(df_features))
if cnn_paths:
    print(f"🎥 Đang gọi {len(cnn_paths)} mô hình 1D-CNN...")
    for p in cnn_paths:
        model = CNN1DModel(num_features=input_dim, hidden_size=128).to(device)
        model.load_state_dict(torch.load(p, map_location=device))
        model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader: # <--- CHẠY TỪNG LÔ NHỎ ĐỂ CHỐNG TRÀN RAM
                preds = model(batch[0]).cpu().numpy()
                fold_preds.append(preds.flatten())
                
        cnn_preds += np.concatenate(fold_preds)
    cnn_preds /= len(cnn_paths)

\n🌳 Đang gọi 20 mô hình LightGBM...
⚡ Đang gọi 3 mô hình MLP...
🎥 Đang gọi 3 mô hình 1D-CNN...


In [9]:
print("\n⚖️ Trộn kết quả theo tỷ lệ SoTA: 45% LGBM | 45% CNN | 10% MLP")
W_LGBM, W_CNN, W_MLP = 0.45, 0.45, 0.10

overall_preds = (lgbm_preds * W_LGBM) + (cnn_preds * W_CNN) + (mlp_preds * W_MLP)

# BÍ MẬT SỐ 2: Ép tất cả các giá trị dự đoán âm (nếu có) về 0
overall_preds = np.clip(overall_preds, 0, None)


⚖️ Trộn kết quả theo tỷ lệ SoTA: 45% LGBM | 45% CNN | 10% MLP


In [10]:
# 7. 📤 KẾT XUẤT SUBMISSION
df_features['target'] = overall_preds
submission = df_features[['row_id', 'target']]

# Kaggle cần ghi file ra thư mục mặc định `/kaggle/working/submission.csv`
submission.to_csv('/kaggle/working/submission.csv', index=False)


print("✅ HOÀN TẤT DỰ ÁN! File submission.csv đã được sinh ra thành công!")
print(submission.head())

✅ HOÀN TẤT DỰ ÁN! File submission.csv đã được sinh ra thành công!
  row_id    target
0    0-4  1.863542
1   0-32  1.862352
2   0-34  1.862352
